# Loading Data

In [97]:
import pandas as pd
import ast
import numpy as np

In [75]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/dnb-handle-data")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\rajat\.cache\kagglehub\datasets\s1nghayush\dnb-handle-data\versions\4


In [98]:
mbm_bowl = pd.read_csv(r'D:\ED\FIFS_IPL\data\mbm\cricsheet_ipl_mbm_bowl.csv')
playing_xi = pd.read_csv(r'D:\ED\FIFS_IPL\data\register_data\playing_xi.csv')
player_info = pd.read_csv(r'D:\ED\FIFS_IPL\data\register_data\player_details.csv')

# Cleaning Data

In [99]:
# Update team names in team1_name and team2_name columns
playing_xi['team1_name'] = playing_xi['team1_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})
playing_xi['team2_name'] = playing_xi['team2_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})

# Also normalize team names in mbm_bowl
mbm_bowl['bowl_team'] = mbm_bowl['bowl_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
})

# Optionally, normalize the batting team if needed
mbm_bowl['bat_team'] = mbm_bowl['bat_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
})


# Standardize bowler_id in mbm_bowl
mbm_bowl['bowler_id'] = mbm_bowl['bowler_id'].astype(str).str.replace(r'\.0$', '', regex=True)


In [100]:
mbm_bowl['bowler_id'] = mbm_bowl['bowler_id'].astype(str)

def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Define a helper function that returns the longest non-null string among the three names
def get_longest_name(row):
    names = [row['cricsheet_name'], row['unique_name'], row['cricbuzz_name']]
    # Filter out any missing values
    names = [name for name in names if pd.notnull(name)]
    return max(names, key=len) if names else ''

# Create a mapping dictionary where the key is the player_id (converted to string)
# and the value is the longest name obtained by applying the function above
player_map = dict(zip(player_info['key_cricinfo'].astype(str),
                      player_info.apply(get_longest_name, axis=1)))

# handling dnb values

In [101]:
dnb_bowl_rows = []

# Group by match_id and innings
for (match_id, inns), group in mbm_bowl.groupby(['match_id', 'inns']):
    # Extract match details
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]   # Batting team
    bowl_team = group['bowl_team'].iloc[0] # Bowling team
    venue = group['venue'].iloc[0]

    # Get set of bowler IDs who actually bowled
    bowled_ids = set(group['bowler_id'].unique())

    # Lookup the playing XI row for the match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]

    # Get the playing XI for the bowling team
    if row['team1_name'] == bowl_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bowl_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Bowling team {bowl_team} not found in playing XI teams for match {match_id}.")
        continue

    # Convert playing XI player IDs to strings
    full_xi = [str(pid) for pid in full_xi]

    # Identify players in playing XI who did not bowl
    dnb_bowlers = [pid for pid in full_xi if pid not in bowled_ids]

    # Add a new row for each bowler who did not bowl
    for pid in dnb_bowlers:
        bowler_name = player_map.get(pid, "")  # Get player name if available
        new_row = {
            'date': date,
            'match_id': match_id,
            'bowl_team': bowl_team,
            'bat_team': bat_team,
            'bowler_id': pid,
            'bowler': bowler_name,
            'inns': inns,
            'overs': 'DNB',
            'runs': 0,
            'wickets': 0,
            'maiden': 0,
            'eco': 0,
            'balls':0,
            'lbw_or_bowled': 0,
            'dots':0,
            'venue': venue
        }
        dnb_bowl_rows.append(new_row)

In [102]:
new_df = pd.DataFrame(dnb_bowl_rows)

In [103]:
combined_df = pd.concat([mbm_bowl, new_df], ignore_index=True)

In [104]:
combined_df['bowler'] = combined_df.apply(
    lambda row: row['bowler'] if pd.notnull(row['bowler']) and row['bowler'] != ""
    else player_map.get(row['bowler_id'], ""), axis=1
)

combined_df['is_DNB_Bowl'] = combined_df['overs'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB_Bowl']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB_Bowl'], inplace=True)


In [105]:
combined_df[combined_df['match_id'] == 335982].sort_values(by='inns')

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,maiden,eco,balls,lbw_or_bowled,dots,venue
0,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,30732,Praveen Kumar,1,4.0,38.0,0,1,9.500000,25,0,14,M Chinnaswamy Stadium
1,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,30102,Zaheer Khan,1,4.0,38.0,1,0,9.500000,24,0,8,M Chinnaswamy Stadium
2,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,6938,Ashley Noffke,1,4.0,40.0,1,0,10.000000,25,0,8,M Chinnaswamy Stadium
3,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,45789,Jacques Kallis,1,4.0,48.0,1,0,12.000000,25,0,10,M Chinnaswamy Stadium
4,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,29725,Sunil Joshi,1,3.0,26.0,0,0,8.666667,18,0,4,M Chinnaswamy Stadium
5,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,8291,Cameron White,1,1.0,24.0,0,0,24.000000,7,0,1,M Chinnaswamy Stadium
6,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,26206,Balachandra Akhil,1,DNB,0.0,0,0,0.000000,0,0,0,M Chinnaswamy Stadium
7,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,44111,Mark Boucher,1,DNB,0.0,0,0,0.000000,0,0,0,M Chinnaswamy Stadium
8,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,28114,Rahul Dravid,1,DNB,0.0,0,0,0.000000,0,0,0,M Chinnaswamy Stadium
9,2008-04-18,335982,Royal Challengers Bangalore,Kolkata Knight Riders,253802,Virat Kohli,1,DNB,0.0,0,0,0.000000,0,0,0,M Chinnaswamy Stadium


In [86]:
combined_df.to_csv(r'D:\ED\FIFS_IPL\data\mbm\cricsheet_ipl_mbm_bowl_final.csv', index=False)